# Session 10 — Your turn: How text becomes numbers

Build the first steps of the text pipeline by hand on a few short reviews: tokens, a vocabulary, ids, and the order-blindness of counting words.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import re
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

reviews = ["the movie was good", "the movie was not good", "a good cast and a good story", "not a good movie at all",
           "the story was bad and the cast was bad", "good fun and a good cast"]
from aij_check import check, close, summary

## Task 1: Tokens, a vocabulary, ids

Write `tokenize(text)` (lower-case words: runs of letters, digits and apostrophes), then build `word_to_id` from all the reviews: the words ordered by how often they occur (most frequent first, ties in alphabetical order),
with id 0 reserved for padding and id 1 for unknown words, so the first real word gets id 2. Write `encode(text, length)` that returns the ids of a sentence cut or padded with 0 to exactly `length` ids (unknown words get 1).

In [2]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())


counts = Counter(token for r in reviews for token in tokenize(r))
ordered = sorted(counts, key=lambda w: (-counts[w], w))
word_to_id = {word: i + 2 for i, word in enumerate(ordered)}


def encode(text, length):
    ids = [word_to_id.get(t, 1) for t in tokenize(text)][:length]
    return ids + [0] * (length - len(ids))

In [3]:
check("tokenize lower-cases and splits", lambda: tokenize("Not GOOD, at all!") == ["not", "good", "at", "all"])
check("the most frequent word, 'good', gets id 2", lambda: word_to_id["good"] == 2)
check("ids are unique and start at 2", lambda: sorted(word_to_id.values()) == list(range(2, 2 + len(word_to_id))))
check("encode pads with 0", lambda: encode("the movie", 5) == [word_to_id["the"], word_to_id["movie"], 0, 0, 0])
check("encode maps unknown words to 1 and cuts long sentences", lambda: encode("zebra good", 1) == [1])
summary()

PASS  tokenize lower-cases and splits
PASS  the most frequent word, 'good', gets id 2
PASS  ids are unique and start at 2
PASS  encode pads with 0
PASS  encode maps unknown words to 1 and cuts long sentences

5 of 5 checks passed.  Well done.


**Why it works.** The ids are labels, not quantities: id 5 is not 'more' than id 3. Padding to a fixed length lets reviews of different sizes sit in one table; the unknown-word id lets the model meet words it never saw.

## Task 2: Counting words forgets the order

With `CountVectorizer()` (single words) compare the vectors of "not good" and "good not": store `same_unigram_vector` (True if they are identical).
Then do the same with `CountVectorizer(ngram_range=(1, 2))`, which also counts word pairs, and store `same_bigram_vector`. Finally set `pair_feature` to the word pair that tells the two sentences apart for the second vectoriser (a string such as `"a b"`) and fit on the six reviews above plus both sentences.

In [4]:
sentences = ["not good", "good not"]
uni = CountVectorizer().fit(reviews + sentences)
a, b = uni.transform(sentences).toarray()
same_unigram_vector = bool((a == b).all())
bi = CountVectorizer(ngram_range=(1, 2)).fit(reviews + sentences)
a2, b2 = bi.transform(sentences).toarray()
same_bigram_vector = bool((a2 == b2).all())
names = bi.get_feature_names_out()
pair_feature = [n for n, x, y in zip(names, a2, b2) if x == 1 and y == 0 and " " in n][0]
print(same_unigram_vector, same_bigram_vector, pair_feature)

True False not good


In [5]:
check("single-word counts cannot tell the sentences apart", lambda: same_unigram_vector is True)
check("word pairs can", lambda: same_bigram_vector is False)
check("the telling pair is 'not good'", lambda: pair_feature == "not good")
summary()

PASS  single-word counts cannot tell the sentences apart
PASS  word pairs can
PASS  the telling pair is 'not good'

3 of 3 checks passed.  Well done.


**Why it works.** A bag of words records which words occur and how often, not where, so 'not good' and 'good not' look identical. Adding word pairs gives the model one feature that only 'not good' switches on, a first, cheap way to bring a little order back in.